# Préparer le corpus de transcriptions

Ce notebook concatène les transcripts scientifiques, workforce et créatifs dans `transcripts.csv`, puis extrait les prises de parole `User:` avec leurs continuations multilignes dans `transcripts_responses.csv`.

## 1. Importer les bibliothèques et définir les chemins

Les chemins fonctionnent si le notebook est lancé depuis `Mistral4/` ou depuis la racine du workspace.

In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

SOURCE_FILES = {
    "scientists": "scientists_transcripts.csv",
    "workforce": "workforce_transcripts.csv",
    "creatives": "creatives_transcripts.csv",
}
data_dir_candidates = [Path("Mistral4"), Path(".")]
DATA_DIR = next(
    (
        path.resolve()
        for path in data_dir_candidates
        if all((path / filename).is_file() for filename in SOURCE_FILES.values())
    ),
    None,
)
if DATA_DIR is None:
    raise FileNotFoundError(
        "Il faut placer scientists_transcripts.csv, workforce_transcripts.csv "
        "et creatives_transcripts.csv dans le même dossier."
    )

SOURCE_PATHS = {name: DATA_DIR / filename for name, filename in SOURCE_FILES.items()}
TRANSCRIPTS_PATH = DATA_DIR / "transcripts.csv"
OUTPUT_PATH = DATA_DIR / "transcripts_responses.csv"
REPORT_PATH = DATA_DIR / "transcripts_preparation_report.csv"

print("Sources :")
for source_name, source_path in SOURCE_PATHS.items():
    print(f"{source_name} : {source_path}")
print(f"Corpus combiné : {TRANSCRIPTS_PATH}")
print(f"Réponses nettoyées : {OUTPUT_PATH}")
print(f"Rapport : {REPORT_PATH}")

## 2. Charger et concaténer les fichiers source

Lire les trois CSV en UTF-8, vérifier leur schéma, puis concaténer les lignes dans `transcripts.csv` en conservant `transcript_id` et `text`.

In [ ]:
source_frames = {}
expected_columns = None
source_counts = {}
for source_name, source_path in SOURCE_PATHS.items():
    frame = pd.read_csv(
        source_path,
        encoding="utf-8",
        sep=",",
        dtype={"transcript_id": "string", "text": "string"},
        keep_default_na=False,
    )
    frame.columns = frame.columns.astype(str).str.strip().str.lower()
    if not {"transcript_id", "text"}.issubset(frame.columns):
        raise ValueError(
            f"{source_path.name} doit contenir les colonnes transcript_id et text. "
            f"Colonnes trouvées : {frame.columns.tolist()}"
        )
    frame = frame[["transcript_id", "text"]].copy()
    if expected_columns is None:
        expected_columns = frame.columns.tolist()
    elif frame.columns.tolist() != expected_columns:
        raise ValueError(f"Schéma incompatible dans {source_path.name}.")
    source_counts[source_name] = len(frame)
    source_frames[source_name] = frame

df_source = pd.concat(source_frames.values(), ignore_index=True)
duplicate_ids = int(df_source["transcript_id"].duplicated(keep=False).sum())
df_source.to_csv(TRANSCRIPTS_PATH, index=False, encoding="utf-8")

print("Lignes par source :", source_counts)
print(f"Total concaténé : {len(df_source):,}")
print(f"Identifiants répétés entre les sources : {duplicate_ids}")
print(f"Corpus brut écrit : {TRANSCRIPTS_PATH}")
df_source.head(3)

## 3. Vérifier la structure et les types

Contrôler l’aperçu, les types, les valeurs manquantes et la longueur des transcripts avant de les transformer.

In [ ]:
print("Dimensions :", df_source.shape)
print("Colonnes :", df_source.columns.tolist())
display(df_source.head(5))
print("\nTypes :")
print(df_source.dtypes)
print("\nValeurs manquantes :")
print(df_source.isna().sum())
print("\nLongueur du transcript (caractères) :")
display(df_source["text"].str.len().describe(percentiles=[0.25, 0.5, 0.75, 0.95]))

## 4. Nettoyer les colonnes et standardiser les noms

Standardiser les en-têtes, supprimer les espaces parasites et homogénéiser les retours à la ligne du texte.

In [ ]:
df = df_source.copy()
df.columns = (
    df.columns.astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", "_", regex=True)
)
required_columns = {"transcript_id", "text"}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"Colonnes obligatoires absentes : {sorted(missing_columns)}")

df = df[["transcript_id", "text"]].copy()
df["transcript_id"] = df["transcript_id"].fillna("").astype("string").str.strip()
df["text"] = (
    df["text"]
    .fillna("")
    .astype("string")
    .str.replace("\r\n", "\n", regex=False)
    .str.replace("\r", "\n", regex=False)
)
df.head(3)

## 5. Traiter les valeurs manquantes et les doublons

Les lignes sans identifiant sont écartées. Les textes manquants deviennent vides ; seuls les doublons exacts (`transcript_id` et `text` identiques) sont supprimés pour ne pas fusionner des contenus distincts partageant un identifiant.

In [ ]:
input_rows = len(df)
missing_id_rows = int(df["transcript_id"].eq("").sum())
missing_text_rows = int(df["text"].str.strip().eq("").sum())
repeated_id_rows = int(df["transcript_id"].duplicated(keep=False).sum())
exact_duplicate_rows = int(df.duplicated(subset=["transcript_id", "text"]).sum())

df = df.loc[df["transcript_id"].ne("")].copy()
df = df.drop_duplicates(subset=["transcript_id", "text"], keep="first").reset_index(drop=True)

print(f"Lignes source : {input_rows:,}")
print(f"Lignes sans identifiant écartées : {missing_id_rows:,}")
print(f"Textes vides conservés : {missing_text_rows:,}")
print(f"Lignes avec un identifiant répété (contenus possiblement distincts) : {repeated_id_rows:,}")
print(f"Doublons exacts supprimés : {exact_duplicate_rows:,}")
print(f"Lignes conservées : {len(df):,}")

## 6. Convertir les formats (dates, nombres, texte)

Aucune date ni donnée numérique métier n’est présente dans ce fichier. On normalise les retours à la ligne et le texte, puis on extrait les tours `User:`. Toute ligne suivante reste rattachée au locuteur courant jusqu’au prochain préfixe reconnu (`User:`, `AI:` ou `Assistant:`).

In [ ]:
SPEAKER_LINE = re.compile(r"^\s*(user|ai|assistant)\s*:\s*(.*)$", flags=re.IGNORECASE)

def extract_user_turns(transcript):
    turns = []
    current_lines = []
    capturing_user = False

    for line in str(transcript).splitlines():
        match = SPEAKER_LINE.match(line)
        if match:
            if capturing_user:
                answer = "\n".join(current_lines).strip()
                if answer:
                    turns.append(answer)

            capturing_user = match.group(1).casefold() == "user"
            current_lines = [match.group(2)] if capturing_user else []
        elif capturing_user:
            current_lines.append(line)

    if capturing_user:
        answer = "\n".join(current_lines).strip()
        if answer:
            turns.append(answer)

    return turns

user_turns = df["text"].map(extract_user_turns)
result = pd.DataFrame({
    "transcript_id": df["transcript_id"].astype("string"),
    "user_responses": user_turns.map(lambda turns: "\n\n".join(turns)),
})

extracted_turn_count = int(user_turns.map(len).sum())
no_response_count = int(result["user_responses"].str.strip().eq("").sum())
print(f"Prises de parole utilisateur extraites : {extracted_turn_count:,}")
print(f"Transcripts sans réponse utilisateur : {no_response_count:,}")
result.head(3)

## 7. Appliquer des règles de validation

Vérifier les champs obligatoires, les identifiants, la correspondance des lignes source/sortie et les doublons d’identifiant. Les éventuelles répétitions non identiques sont signalées sans être supprimées.

In [ ]:
validation_rows = [
    {
        "controle": "Colonnes de sortie attendues",
        "valide": set(result.columns) == {"transcript_id", "user_responses"},
        "detail": ", ".join(result.columns),
    },
    {
        "controle": "Aucun identifiant vide",
        "valide": bool(result["transcript_id"].str.strip().ne("").all()),
        "detail": f"{int(result['transcript_id'].str.strip().eq('').sum())} identifiant(s) vide(s)",
    },
    {
        "controle": "Nombre de lignes cohérent après nettoyage",
        "valide": len(result) == len(df),
        "detail": f"{len(result)} sortie(s) pour {len(df)} ligne(s) conservée(s)",
    },
    {
        "controle": "Unicité des identifiants",
        "valide": bool(result["transcript_id"].is_unique),
        "detail": f"{int(result['transcript_id'].duplicated(keep=False).sum())} ligne(s) avec identifiant répété",
    },
]
validation_report = pd.DataFrame(validation_rows)
display(validation_report)

failed_checks = validation_report.loc[~validation_report["valide"]]
if not failed_checks.empty:
    print("À examiner avant analyse :")
    display(failed_checks)

## 8. Exporter le fichier préparé et le rapport de contrôle

Afficher un exemple avant/après, puis écrire les réponses extraites et le rapport dans deux nouveaux fichiers CSV, sans écraser la source.

In [ ]:
if len(df) > 0:
    example_position = 0
    print("Transcript source (extrait) :")
    print(df.iloc[example_position]["text"][:1000])
    print("\nRéponse(s) utilisateur extraite(s) :")
    print(result.iloc[example_position]["user_responses"][:1000])

average_turns_per_transcript = (
    float(np.mean(user_turns.map(len))) if len(user_turns) else np.nan
)
report_rows = [
    {"controle": "Fichiers sources", "valeur": "; ".join(str(path) for path in SOURCE_PATHS.values())},
    {"controle": "Corpus concaténé", "valeur": str(TRANSCRIPTS_PATH)},
    {"controle": "Réponses exportées", "valeur": str(OUTPUT_PATH)},
    {"controle": "Lignes par source", "valeur": str(source_counts)},
    {"controle": "Lignes source concaténées", "valeur": input_rows},
    {"controle": "Lignes sans identifiant écartées", "valeur": missing_id_rows},
    {"controle": "Textes vides conservés", "valeur": missing_text_rows},
    {"controle": "Doublons exacts supprimés", "valeur": exact_duplicate_rows},
    {"controle": "Lignes à identifiant répété conservées", "valeur": repeated_id_rows},
    {"controle": "Transcripts exportés", "valeur": len(result)},
    {"controle": "Prises de parole utilisateur extraites", "valeur": extracted_turn_count},
    {"controle": "Transcripts sans réponse utilisateur", "valeur": no_response_count},
    {"controle": "Nombre moyen de réponses par transcript", "valeur": average_turns_per_transcript},
    {"controle": "Contrôles tous valides", "valeur": bool(validation_report["valide"].all())},
]
preparation_report = pd.DataFrame(report_rows)

data_to_export = result.copy()
data_to_export.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")
preparation_report.to_csv(REPORT_PATH, index=False, encoding="utf-8")

print(f"\nCorpus concaténé : {TRANSCRIPTS_PATH}")
print(f"Réponses exportées : {OUTPUT_PATH}")
print(f"Rapport enregistré : {REPORT_PATH}")
display(preparation_report)